# Neural-Network Magnitude Prediction

## What this notebook does
This notebook loops through every CSV file in a Google Drive input folder, imputes missing predictor values, removes highly correlated predictors, standardizes the predictors and response, tunes a dense neural network with Optuna, evaluates it on held-out data, and saves one results table per dataset.

## Required inputs
- Google Colab with access to Google Drive.
- One or more CSV files stored directly in `input_folder`.
- Each CSV must contain the columns expected by the preprocessing section described below.

## Outputs
- One `<dataset>_ANN_results.csv` file per input CSV, containing test R-squared, MAE, RMSE, MAPE, and the best hyperparameters.
- Training histories and progress messages shown in the notebook output.

## Settings a new user should review
- `input_folder`: change this to the Google Drive folder containing the input CSV files.
- `output_folder`: change this to the folder where result CSV files should be written.
- `df.drop("site_id", axis=1)`: edit or remove this line if your identifier column has a different name or is absent.
- `df = df[df.columns[1:]]`: edit or remove this line if the first remaining column should not be discarded.
- The correlation threshold (`0.9`), split proportions, random seed, and number of Optuna trials can be changed if required.

## Important data assumptions
- After `site_id` is removed, the script discards the first remaining column.
- The first column left after those removals is treated as the response variable; all later columns are predictors.
- The input folder should contain only CSV files that follow this same structure, because every CSV in the folder is processed.

Run the notebook from top to bottom in Google Colab. The progress messages explain what is happening and where outputs are written.

## 1. Install required packages
Run this cell once at the start of a fresh Colab session.

In [ ]:
# Install Packages
!pip install keras-tuner tensorflow pandas numpy scikit-learn shap
!pip install optuna --quiet

## 2. Import packages

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import optuna
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from optuna.pruners import MedianPruner
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import BatchNormalization

# Mount Google Drive so the notebook can read inputs and save outputs
from google.colab import drive

## 3. Mount Google Drive and set directories
**Edit `input_folder` and `output_folder` in the next cell before running the notebook on a new system.**

In [ ]:
drive.mount('/content/drive')

# User configuration: input and output directories
input_folder = '/content/drive/My Drive/machine_learning_colab/'
output_folder = '/content/drive/My Drive/machine_learning_colab/outputs_new/'
os.makedirs(output_folder, exist_ok=True)

## 4. Process each dataset, tune the model, and save results

In [ ]:
for file_name in os.listdir(input_folder):
    if file_name.endswith('.csv'):
        dataset_path = os.path.join(input_folder, file_name)
        dataset_name = file_name.replace('.csv', '')
        print(f"\nStarting neural-network modelling for dataset: {dataset_name}")

        # Load dataset
        df = pd.read_csv(dataset_path)
        print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns from {file_name}.")
        df = df.drop('site_id', axis=1)
        df = df[df.columns[1:]]

        # Removal of variables with high correlation
        corr_matrix = df.iloc[:, 1:].corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
        df = df.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} predictors with absolute pairwise correlation above 0.90.")

        # Separate response and predictors
        Y = df.iloc[:, 0].values.reshape(-1, 1)
        X = df.iloc[:, 1:]
        feature_names = X.columns.tolist()

        # Handle NaNs with mean imputation
        imputer = SimpleImputer(strategy='mean')
        X = imputer.fit_transform(X)

        mask = ~np.isnan(Y).flatten()
        X = X[mask]
        Y = Y[mask]

        # Standardization
        scaler_x = StandardScaler()
        scaler_y = StandardScaler()
        X_scaled = scaler_x.fit_transform(X)
        Y_scaled = scaler_y.fit_transform(Y).flatten()

        # Split data: 60% train, 20% validation, 20% test
        X_train, X_temp, Y_train, Y_temp = train_test_split(X_scaled, Y_scaled, test_size=0.4, random_state=42)
        X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

        # Optuna objective function
        def objective(trial):
            n_layers = trial.suggest_int('n_layers', 1, 5)
            layer_units = [trial.suggest_int(f'n_units_layer{i}', 32, 512) for i in range(n_layers)]
            dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
            learning_rate = trial.suggest_loguniform('learning_rate', 1e-4, 1e-2)
            l2_reg = trial.suggest_float('l2_reg', 1e-6, 1e-3, log=True)

            model = Sequential()
            model.add(Dense(layer_units[0], activation='relu', input_shape=(X_train.shape[1],),
                            kernel_regularizer=l2(l2_reg)))
            for units in layer_units[1:]:
                model.add(Dense(units, activation='relu', kernel_regularizer=l2(l2_reg)))
                model.add(Dropout(dropout_rate))
            model.add(Dense(1, activation='linear', kernel_regularizer=l2(l2_reg)))

            optimizer = Adam(learning_rate=learning_rate)
            model.compile(optimizer=optimizer, loss='mean_squared_error')

            history = model.fit(
                X_train, Y_train,
                validation_data=(X_val, Y_val),
                epochs=100,
                batch_size=32,
                verbose=0,
                callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)]
            )

            val_preds_scaled = model.predict(X_val).flatten()
            val_preds_unscaled = scaler_y.inverse_transform(val_preds_scaled.reshape(-1, 1)).flatten()
            val_true_unscaled = scaler_y.inverse_transform(Y_val.reshape(-1, 1)).flatten()
            rmse = np.sqrt(mean_squared_error(val_true_unscaled, val_preds_unscaled))
            return rmse

        # Add early stopping pruner
        pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5, interval_steps=1)

        # Run hyperparameter optimization
        print("Starting Optuna optimization with 50 trials. This may take some time for larger datasets.")
        study = optuna.create_study(direction='minimize', pruner=pruner)
        study.optimize(objective, n_trials=50, show_progress_bar=True)

        # Build final ANN model with best parameters
        best_params = study.best_trial.params
        print(f"Optimization finished. Best parameters: {best_params}")
        n_layers = best_params['n_layers']
        layer_units = [best_params[f'n_units_layer{i}'] for i in range(n_layers)]
        dropout_rate = best_params['dropout_rate']
        learning_rate = best_params['learning_rate']
        l2_reg = best_params['l2_reg']

        model = Sequential()
        model.add(Dense(layer_units[0], activation='relu', input_shape=(X_train.shape[1],),
                        kernel_regularizer=l2(l2_reg)))
        for units in layer_units[1:]:
            model.add(Dense(units, activation='relu', kernel_regularizer=l2(l2_reg)))
            model.add(Dropout(dropout_rate))
        model.add(Dense(1, activation='linear', kernel_regularizer=l2(l2_reg)))

        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mean_squared_error')

        # Train final model with early stopping on validation set
        model.fit(
            X_train, Y_train,
            validation_data=(X_val, Y_val),
            epochs=200,
            batch_size=32,
            verbose=1,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)]
        )

        # Predict on test set
        test_preds = model.predict(X_test).flatten()

        # Unstandardize response for true RMSE/MAE/MAPE values
        unstand_preds = scaler_y.inverse_transform(test_preds.reshape(-1, 1)).flatten()
        unstand_true = scaler_y.inverse_transform(Y_test.reshape(-1, 1)).flatten()

        def evaluate(y_true, y_pred, label='Test'):
            test_r2 = r2_score(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mape = mean_absolute_percentage_error(y_true, y_pred)
            return test_r2, mae, rmse, mape

        test_r2, mae, rmse, mape = evaluate(unstand_true, unstand_preds, label='Test')

        # Save results to CSV
        results_df = pd.DataFrame([{
            "Dataset": dataset_name,
            "Test_R2": test_r2,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape,
            "Best_Params": best_params
        }])

        output_file = os.path.join(output_folder, f"{dataset_name}_ANN_results.csv")
        results_df.to_csv(output_file, index=False)
        print(f"Finished {dataset_name}. Results were saved to: {output_file}")